In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Final merged dataset

In [ ]:
df=pd.read_csv('final_merged_data.csv')

In [ ]:
df.head()

### EDA overview
1. data overview
2. Sales performance analysis
3. Weekly, monthly, etc sales trend
4. Holiday analysis
5. Store Analysis
6. Department analysis
7. external factors
8. Markdown analysis
9. Correlation analysis
10. Distribution analysis
11. Outlier analysis
12. Business Insights

### Data overview

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.rename(columns={'IsHoliday': 'Holiday'}, inplace=True)

In [ ]:
df.columns

In [ ]:
df['Date']=pd.to_datetime(df['Date'])

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
#Sales performance analysis
df['Weekly_Sales'].describe().round()

### Interpretation
- there are 420285.0 weekly sales in dataset
- the average weekly sales is 16030.0
- 22729.0 is the sd, meaning sales vary a lot across stores
- Min sales is 0, probably the stores were closed for the week
- 25% of the data have weekly sales below 2118.0
- 50% of the data, which is also the median, have weekly sales below 7659.0
- 75% of the data have weekly sales below 20268.0
- the highest recorded weekly sales is 693099.0

In [ ]:
sns.histplot(df['Weekly_Sales'], kde=True, bins=50,)
plt.ylabel('frequency')
plt.show()

In [ ]:
# the histogram is positively skewed, meaning
# some stores/depts have exceptionally large sales observations 

In [ ]:
sns.histplot(np.log1p(df['Weekly_Sales']), kde=True, bins=50,)
plt.ylabel('frequency')
plt.xlabel("Log(Weekly_Sales)")
plt.show()

### Store performance

In [ ]:
df.columns

In [ ]:
# Top 10 stores by sales
Top_10_stores=df.groupby('Store')['Weekly_Sales'].sum().sort_values(ascending=False)
Top_10_stores.head(10)

In [ ]:
pd.options.display.float_format='{:,.0f}'.format

In [ ]:
# Bottom 10 stores by sales
Bottom_10_stores=df.groupby('Store')['Weekly_Sales'].sum().round(0).sort_values(ascending=True)
Bottom_10_stores.head(10)

In [ ]:
# Storewise average weekly sales
Storewise_avg_weekly_sales=df.groupby('Store')['Weekly_Sales'].mean().sort_values(ascending=False).reset_index()
Storewise_avg_weekly_sales.rename(columns={'Weekly_Sales': 'Mean_Weekly_Sales'}, inplace=True)
Storewise_avg_weekly_sales.head()

In [ ]:
plt.figure(figsize=(18,10))
sns.barplot(data=Storewise_avg_weekly_sales, x='Store', y='Mean_Weekly_Sales', order=Storewise_avg_weekly_sales['Store'] )
plt.xlabel('Store')
plt.ylabel('Mean_Weekly_Sales')
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Store-wise sales variability (standard deviation)
Store_wise_SD=df.groupby('Store')['Weekly_Sales'].std().round().sort_values(ascending=False).reset_index()
Store_wise_SD.rename(columns={'Weekly_Sales': 'SD_Weekly_Sales'}, inplace=True)
Store_wise_SD.head()

In [ ]:
Store_stats=pd.merge(Store_wise_SD, Storewise_avg_weekly_sales, on='Store')
Store_stats.head()

In [ ]:
Store_stats['CV']=(Store_stats['SD_Weekly_Sales']/Store_stats['Mean_Weekly_Sales']).round(2)
Store_stats.sort_values(by='CV', ascending=True).head()

In [ ]:
# The sales of store 23 is the most stable

In [ ]:
Store_stats.dtypes

In [ ]:
pd.options.display.float_format='{:,.2f}'.format

### Department Performance

In [ ]:
df.columns

In [ ]:
df.Dept.nunique()

In [ ]:
df.Dept.unique()

In [ ]:
# Top selling departments
dept_aggregates=df.groupby('Dept')['Weekly_Sales'].agg(['sum', 'std', 'mean'])

In [ ]:
dept_aggregates['sum'].sort_values(ascending=False).head(10)

In [ ]:
# Bottom selling department
dept_aggregates['sum'].sort_values(ascending=True).head(10)

In [ ]:
total_sales=df['Weekly_Sales'].sum()

In [ ]:
# Dept wise contribution to total sales
Dept_wise_contribution= (dept_aggregates['sum']/total_sales)*100
Top_dep= (Dept_wise_contribution.sort_values(ascending=False).head(10))
Top_dep

In [ ]:
#Other depts
others= Dept_wise_contribution.sort_values(ascending=False).iloc[10:].sum().round()
others

In [ ]:
#combined depts
Pie_chart=Top_dep
Pie_chart['others']=others
Pie_chart

In [ ]:
plt.figure(figsize=(9,9))
explode= [0.1]*10 +[0]
plt.pie(Pie_chart, labels= Pie_chart. index, autopct='%1.1f%%', explode=explode,startangle=90, wedgeprops={'edgecolor':'white','linewidth':1})
plt.title('Dept contribution to total sales')
plt.show()

In [ ]:
#Department-wise variability
Dept_agg=df.groupby('Dept')['Weekly_Sales'].agg(['sum', 'mean', 'std'])
Dept_agg.head()

In [ ]:
Depwise_CV= (Dept_agg['std']/Dept_agg['mean']).sort_values(ascending=False).round(2)
Depwise_CV.head()

In [ ]:
Dept_agg['CV']= Depwise_CV
Dept_agg.sort_values(by='CV',ascending=False).round(2).head()

In [ ]:
#Dept 59 is the most stable dept

### Store Type Performance

In [ ]:
#Average sales by Type (A/B/C)

In [ ]:
df.columns

In [ ]:
Sales_by_type=df.groupby('Type')['Weekly_Sales'].agg(['sum', 'mean', 'std'])
Sales_by_type

In [ ]:
Sales_dist_by_type=(Sales_by_type['sum']/total_sales)*100
Sales_dist_by_type

In [ ]:
Sales_by_type['Type_Sales_contribution']=Sales_dist_by_type
Sales_by_type

In [ ]:
#Number of stores in each Type 

In [ ]:
Store_count=(df.groupby('Type')['Store'].nunique().reset_index())
Store_count.columns=['Store_Type', 'Number_of_stores']
Store_count.style.hide(axis='index')

### Store Size Performance

In [ ]:
df.columns

In [ ]:
df.Size.nunique()

In [ ]:
store_size_sales=df.groupby('Size')['Weekly_Sales'].sum().reset_index()
store_size_sales.sort_values(by='Size', ascending=False).head()

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(data=store_size_sales,x='Size',y='Weekly_Sales',s=100)

plt.title("Relationship Between Store Size and total Weekly Sales")
plt.xlabel("Store Size")
plt.ylabel("Total Weekly Sales")

plt.show()

In [ ]:
#correlation between size and sales
corr=store_size_sales.corr()
corr

### There is a strong positive relationship between store size and weekly sales. Larger stores generally generate higher weekly sales. But we cannot conclude that size alone determines sales. Stores arou d 200,000 sq.feet have sales ranging from 170 to 400 million. So other factors can also contribute such as location, store type, promotions, holidays, unempleyment, CPI and fuel prices.

### Time-Based Performance

In [ ]:
df['Year']=df['Date'].dt.year
df['Month_name']=df['Date'].dt.month_name()
df['Month']=df['Date'].dt.month
df['Quarter']=df['Date'].dt.quarter

In [ ]:
#Weekly sales
weekly_sales=df.groupby('Date')['Weekly_Sales'].sum().reset_index()
weekly_sales.head()

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(weekly_sales['Date'], weekly_sales['Weekly_Sales'])
plt.title("Weekly Sales Trend")
plt.xlabel("Date")
plt.ylabel("Weekly Sales")
plt.show()

In [ ]:
weekly_sales=weekly_sales.sort_values(by='Weekly_Sales', ascending=False)
weekly_sales

In [ ]:
#Monthly sales trend
monthly_sales=df.groupby(['Month_name', 'Month'])['Weekly_Sales'].sum().reset_index()

In [ ]:
plt.figure(figsize=(14,8))
sns.barplot(data=monthly_sales, x='Month_name', y='Weekly_Sales')
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")

plt.show()

In [ ]:
monthly_sales= monthly_sales.sort_values(by='Weekly_Sales', ascending=False)
monthly_sales

In [ ]:
# Quaterly sales
quaterly_sales=df.groupby(['Year','Quarter'])['Weekly_Sales'].sum().reset_index()
quaterly_sales

In [ ]:
sns.barplot(data=quaterly_sales, x='Year', y='Weekly_Sales', hue='Quarter')
plt.title("Quaterly Sales Trend")
plt.xlabel("Qtr")
plt.ylabel("Sales")
plt.show()

In [ ]:
quaterly_sales=quaterly_sales.sort_values(by='Weekly_Sales', ascending=False)
quaterly_sales

In [ ]:
#Yearly trend
Yearly_sales=df.groupby('Year')['Weekly_Sales'].sum().reset_index()
Yearly_sales

In [ ]:
plt.plot(Yearly_sales['Year'],Yearly_sales['Weekly_Sales'], marker='o' )
plt.title("Yearly Sales Trend")
plt.xlabel("Year")
plt.ylabel("Sales")
plt.xticks(Yearly_sales['Year'])
plt.grid(alpha=0.3)
plt.show()

### Holiday Performance

In [ ]:
# Holiday vs non-holiday sales

In [ ]:
df.columns

In [ ]:
holiday_sales=df.groupby('Holiday')['Weekly_Sales'].sum().reset_index()
holiday_sales

In [ ]:
sns.barplot(data=holiday_sales, x=holiday_sales['Holiday'], y=holiday_sales['Weekly_Sales'])
plt.show()

In [ ]:
holiday_df=df[df['Holiday']==True]

In [ ]:
holiday_sales_storewise=holiday_df.groupby(['Holiday', 'Store'])['Weekly_Sales'].sum().sort_values(ascending=False).reset_index()
holiday_sales_storewise.head()

In [ ]:
holiday_sales_deptwise=holiday_df.groupby(['Holiday', 'Dept'])['Weekly_Sales'].sum().sort_values(ascending=False).reset_index()
holiday_sales_deptwise.head()

In [ ]:
#Average holiday sales 
avg_holiday_sales=holiday_df.groupby('Holiday')['Weekly_Sales'].mean().reset_index()
avg_holiday_sales

In [ ]:
#Average non-holiday sales
non_holiday_df=df[df['Holiday']==False]

In [ ]:
avg_non_holiday_sales=non_holiday_df.groupby('Holiday')['Weekly_Sales'].mean().reset_index()
avg_non_holiday_sales

### Promotion Performance

In [ ]:
promo_df=df[
(df['MarkDown1']>0) |
(df['MarkDown2']>0)|
(df['MarkDown3']>0)|
(df['MarkDown4']>0)|
(df['MarkDown5']>0)]

In [ ]:
promo_df.head(2)

In [ ]:
promo_sales=promo_df['Weekly_Sales'].sum()
promo_sales

In [ ]:
non_promo_df=df[
(df['MarkDown1']==0) |
(df['MarkDown2']==0)|
(df['MarkDown3']==0)|
(df['MarkDown4']==0)|
(df['MarkDown5']==0)]

In [ ]:
non_promo_sales=non_promo_df['Weekly_Sales'].sum()
non_promo_sales

In [ ]:
sales_compare=pd.DataFrame({'category':['Promo','No_promo'],
                          'Sales': [promo_sales, non_promo_sales]})
sales_compare.style.hide(axis='index')

In [ ]:
sns.barplot(data=sales_compare,
            x='category',
            y='Sales')

plt.title("Sales During Promotion vs No Promotion")
plt.show()

In [ ]:
#Which MarkDown variable has the strongest relationship with sales?
corr=df[['Weekly_Sales','MarkDown1', 'MarkDown2','MarkDown3', 'MarkDown4', 'MarkDown5']].corr()
corr['Weekly_Sales'].sort_values(ascending=False)

### the correlation with Markdown5 and 1 is very weak. This indicates that promotional discounts donot influence sales that much

In [ ]:
## sales that happened during promotions
sales_during_promotions=promo_df['Weekly_Sales'].sum()
sales_during_promotions

In [ ]:
#promo impact
promo_impact=((sales_during_promotions/total_sales)*100).round()
promo_impact

### Around 36% total sales occurred during promotional weeks

### External Factors

In [ ]:
#Temperature , Fuel Price , CPI ,Unemployment 

In [ ]:
external_factor_corr=df[['Weekly_Sales',
                    'Temperature',
                    'Fuel_Price',
                    'CPI',
                    'Unemployment']].corr()
external_factor_corr['Weekly_Sales'].sort_values(ascending=False)

In [ ]:
##Subplot
plt.figure(figsize=(16,10))
plt.subplot(2,2,1)
sns.scatterplot(data=df,
                x='Temperature',
                y='Weekly_Sales',
               s=10)
plt.title("Temperature vs Weekly Sales")

plt.subplot(2,2,2)
sns.scatterplot(data=df,
                x='Fuel_Price',
                y='Weekly_Sales',
               s=10)

plt.title("Fuel Price vs Weekly Sales")

plt.subplot(2,2,3)
sns.scatterplot(data=df,
                x='CPI',
                y='Weekly_Sales',
               s=10)
plt.title("CPI vs Weekly Sales")

plt.subplot(2,2,4)
sns.scatterplot(data=df,
                x='Unemployment',
                y='Weekly_Sales',
               s=10)
plt.title("Unemployment vs Weekly Sales")
plt.show()

### 
Temperature shows virtually no relationship.
Fuel Price has no meaningful relationship.
CPI has a negligible negative relationship.
Unemployment has a negligible negative relationship.

In [ ]:
df.columns

In [ ]:
### Saving df_final to csv
df.to_csv('df_final.csv', index=False)